In [0]:
from pyspark.sql.functions import col

df_txn = spark.table("databricks_banking_dev_ws2.silver.customer_transactions_clean")
df_branch = spark.table("databricks_banking_dev_ws2.silver.branch_reference_clean")

df_txn.display()

In [0]:
from pyspark.sql.functions import count, sum as _sum, avg, countDistinct, round as _round

# Aggregate transactions by customer location (acting as proxy for branch city)
df_location_summary = (
    df_txn.groupBy("CustLocation")
    .agg(
        count("TransactionID").alias("Total_Transactions"),
        countDistinct("CustomerID").alias("Unique_Customers"),
        _round(_sum("TransactionAmount"), 2).alias("Total_Transaction_Value"),
        _round(avg("TransactionAmount"), 2).alias("Avg_Transaction_Amount"),
        _round(avg("CustAccountBalance"), 2).alias("Avg_Account_Balance")
    )
)

df_location_summary.display()

In [0]:
from pyspark.sql.functions import col, round as _round, create_map, lit, sum as _sum, avg
from itertools import chain

# Map top customer locations to your 12 Canadian branches
location_to_branch = {
    "MUMBAI": "BR001", "BANGALORE": "BR002", "NEW DELHI": "BR003", "DELHI": "BR004",
    "NAVI MUMBAI": "BR005", "LUCKNOW": "BR006", "FARIDABAD": "BR007", "MOHALI": "BR008",
    "SAHARANPUR": "BR009", "THRISSUR": "BR010", "NELLORE": "BR011", "GULBARGA": "BR012",
    "ROURKELA": "BR001", "JAUNPUR": "BR002"
}

mapping_expr = create_map([lit(x) for x in chain(*location_to_branch.items())])

df_mapped = df_location_summary.withColumn("Branch_ID", mapping_expr[col("CustLocation")]).filter(col("Branch_ID").isNotNull())

df_branch_performance = (
    df_mapped.join(df_branch, on="Branch_ID", how="inner")
    .groupBy("Branch_ID", "Branch_Name", "City", "Province", "Region", "Branch_Type", "Staff_Count")
    .agg(
        _sum("Total_Transactions").alias("Total_Transactions"),
        _sum("Unique_Customers").alias("Unique_Customers"),
        _round(_sum("Total_Transaction_Value"), 2).alias("Total_Transaction_Value"),
        _round(avg("Avg_Transaction_Amount"), 2).alias("Avg_Transaction_Amount")
    )
    .withColumn("Transactions_Per_Staff", _round(col("Total_Transactions") / col("Staff_Count"), 2))
)

df_branch_performance.display()

In [0]:
df_branch_performance.write.format("delta").mode("overwrite").saveAsTable(
    "databricks_banking_dev_ws2.gold.branch_performance_summary"
)

print("Gold table created successfully!")